In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path("../datasets/student_resource/student_resource/dataset")

TRAIN_S1 = BASE / "train" / "train_source1.tsv"
TRAIN_GT = BASE / "train" / "train_ground_truth.tsv"

SPLIT_DIR = Path("../data/splits")

SEED, TRAIN_FRAC = 42, 0.80


In [4]:
_dataset_rel = Path("datasets/student_resource/student_resource/dataset")
_candidates = []

_existing_s1 = globals().get("TRAIN_S1")
if _existing_s1 is not None:
	_existing_s1 = Path(_existing_s1).expanduser()
	if _existing_s1.is_file():
		_candidates.append(_existing_s1.resolve().parent.parent)

for _root in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
	_candidates.append(_root / _dataset_rel)

for _search_root in (Path.cwd().resolve(), Path.cwd().resolve().parent):
	for _match in _search_root.glob("**/train/train_source1.tsv"):
		_candidates.append(_match.resolve().parent.parent)

BASE = next(
	(p.resolve() for p in _candidates if (p / "train" / "train_source1.tsv").is_file()),
	None,
)
if BASE is None:
	raise FileNotFoundError(
		f"Could not find train_source1.tsv. Check the dataset path; notebook working directory: {Path.cwd()}"
	)

TRAIN_S1 = BASE / "train" / "train_source1.tsv"
TRAIN_GT = BASE / "train" / "train_ground_truth.tsv"

s1 = pd.read_csv(TRAIN_S1, sep="\t", dtype="string", keep_default_na=True)
gt = pd.read_csv(TRAIN_GT, sep="\t", dtype="string", keep_default_na=False)
print(f"S1 rows: {len(s1):,} | GT rows: {len(gt):,}")
print("S1 cols:", list(s1.columns), "| GT cols:", list(gt.columns))
print(gt.head(2).to_string())


S1 rows: 2,206,821 | GT rows: 2,206,821
S1 cols: ['entity_id', 'business_name', 'business_address', 'country'] | GT cols: ['source1_entity_id', 'matched_entity_ids']
  source1_entity_id                                               matched_entity_ids
0         S1-965667  S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364
1       S1-55344266              S2-249013014,S2-197070651,S3-478195123,S3-384364074


In [5]:
# Audit-only: report null/empty per column. NO rows are dropped —
# the PDF treats missing components as expected noise, and every S1
# entity must keep exactly one GT row.
for c in s1.columns:
    col = s1[c]
    bad = col.isna() | (col.astype("string").str.strip() == "")
    print(f"{c}: null/empty={int(bad.fillna(True).sum()):,}")

s1_ids = s1["entity_id"]
print(f"S1 rows: {len(s1):,} | GT rows: {len(gt):,} (nothing dropped)")

s1_set, gt_set = set(s1_ids.tolist()), set(gt["source1_entity_id"].tolist())
print(f"unique S1: {len(s1_set):,} | unique GT ids: {len(gt_set):,}")
print("dup S1 ids:", int(s1_ids.duplicated().sum()), "| dup GT ids:", int(gt['source1_entity_id'].duplicated().sum()))
missing_in_gt = s1_set - gt_set
missing_in_s1 = gt_set - s1_set
print("S1 w/o GT row:", len(missing_in_gt), "| GT w/o S1 row:", len(missing_in_s1))
assert not missing_in_gt and not missing_in_s1, "S1<->GT mismatch — stop before splitting"
assert len(s1_set) == len(gt) == len(s1_ids), "expected 1 GT row per S1 entity"
print("PASS: every S1 entity has exactly one GT row")


entity_id: null/empty=0
business_name: null/empty=0
business_address: null/empty=0
country: null/empty=0
S1 rows: 2,206,821 | GT rows: 2,206,821 (nothing dropped)
unique S1: 2,206,821 | unique GT ids: 2,206,821
dup S1 ids: 0 | dup GT ids: 0
S1 w/o GT row: 0 | GT w/o S1 row: 0
PASS: every S1 entity has exactly one GT row


In [6]:
rng = np.random.RandomState(SEED)
ids = np.array(sorted(s1_set))
rng.shuffle(ids)
cut = int(len(ids) * TRAIN_FRAC)
train_ids, val_ids = set(ids[:cut].tolist()), set(ids[cut:].tolist())
assert not (train_ids & val_ids) and len(train_ids) + len(val_ids) == len(s1_set)
print(f"train S1: {len(train_ids):,} | val S1: {len(val_ids):,} | seed={SEED}")


train S1: 1,765,456 | val S1: 441,365 | seed=42


In [7]:
m = gt["matched_entity_ids"].fillna("")
n_match = (m != "").astype(int) * (m.str.count(",") + 1)
gt["split"] = np.where(gt["source1_entity_id"].isin(train_ids), "train", "val")
for sp in ["train", "val"]:
    v = n_match[gt["split"] == sp]
    print(f"\n[{sp}] n={len(v):,} singleton_rate={float((v == 0).mean()):.4f} mean={v.mean():.2f}")
    print(v.value_counts().sort_index().head(12).to_string())



[train] n=1,765,456 singleton_rate=0.0559 mean=3.46
matched_entity_ids
0      98631
1      95201
2     300024
3     424555
4     387409
5     257708
6     131851
7      51279
8      15016
9       3320
10       432
11        30

[val] n=441,365 singleton_rate=0.0558 mean=3.46
matched_entity_ids
0      24616
1      23956
2      75188
3     106286
4      96706
5      64249
6      33017
7      12689
8       3664
9        885
10       102
11         7


In [8]:
import re as _re

_WS = _re.compile(r"\s+")
_NON_ALNUM = _re.compile(r"[^a-z0-9 ]")
_NON_ALNUM_NOSPACE = _re.compile(r"[^a-z0-9]")
_MULTI_SP = _re.compile(r" +")

_NAME_CANON = {
    "corporation": "corp", "incorporated": "inc", "company": "co",
    "private": "pvt", "limited": "ltd", "&": "and",
}
_ADDR_CANON = {
    "street": "st", "avenue": "ave", "road": "rd", "boulevard": "blvd",
    "drive": "dr", "lane": "ln", "place": "pl", "court": "ct", "circle": "cir",
}

def _safe(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    try:
        if pd.isna(s):
            return ""
    except Exception:
        pass
    return str(s)

def _basic(s):
    s = _safe(s).lower().replace("&", " and ")
    s = _NON_ALNUM.sub(" ", s)
    return _MULTI_SP.sub(" ", _WS.sub(" ", s)).strip()

def normalize_name(s):
    t = _basic(s)
    if not t:
        return ""
    toks = [_NAME_CANON.get(w, w) for w in t.split(" ")]
    return " ".join(toks)

def normalize_address(s):
    t = _basic(s)
    if not t:
        return ""
    return " ".join(_ADDR_CANON.get(w, w) for w in t.split(" "))

def compact(s):
    return _NON_ALNUM_NOSPACE.sub("", _safe(s).lower())

def add_norm_columns(df):
    df = df.copy()
    df["name_norm"] = df["business_name"].map(normalize_name).astype("string")
    df["addr_norm"] = df["business_address"].map(normalize_address).astype("string")
    df["name_compact"] = df["name_norm"].str.replace(" ", "", regex=False).astype("string")
    df["addr_compact"] = df["addr_norm"].str.replace(" ", "", regex=False).astype("string")
    return df



In [9]:
s1 = add_norm_columns(s1)
print("s1 cols:", list(s1.columns))

samp = s1[["business_name", "name_norm", "name_compact", "business_address", "addr_norm"]].sample(8, random_state=42)
for _, r in samp.iterrows():
    print("-" * 100)
    print("NAME:", r["business_name"], "\n  -> norm:", r["name_norm"], "\n  -> compact:", r["name_compact"])
    print("ADDR:", r["business_address"], "\n  -> norm:", r["addr_norm"])

for c_raw, c_norm in [("business_name", "name_norm"), ("business_address", "addr_norm")]:
    ur, un = s1[c_raw].nunique(), s1[c_norm].nunique()
    dr = float(s1[c_raw].duplicated(keep=False).mean())
    dn = float(s1[c_norm].duplicated(keep=False).mean())
    print(f"{c_raw}: nunique raw={ur:,} norm={un:,} (delta={un-ur:+,}) | dup_rate raw={dr:.4f} norm={dn:.4f}")


s1 cols: ['entity_id', 'business_name', 'business_address', 'country', 'name_norm', 'addr_norm', 'name_compact', 'addr_compact']
----------------------------------------------------------------------------------------------------
NAME: Pediatric Medicine PLLC 
  -> norm: pediatric medicine pllc 
  -> compact: pediatricmedicinepllc
ADDR: 4850 20, Otisco, NY 
  -> norm: 4850 20 otisco ny
----------------------------------------------------------------------------------------------------
NAME: Fetech National Twin 
  -> norm: fetech national twin 
  -> compact: fetechnationaltwin
ADDR: 19034 Woodburn Road, Woodburn, IN 
  -> norm: 19034 woodburn rd woodburn in
----------------------------------------------------------------------------------------------------
NAME: General Design Innovations LLC 
  -> norm: general design innovations llc 
  -> compact: generaldesigninnovationsllc
ADDR: 7241 Osage Avenue, Mesa, AZ 
  -> norm: 7241 osage ave mesa az
-----------------------------------------

In [10]:
TRAIN_S2 = BASE / "train" / "train_source2.tsv"
TRAIN_S3 = BASE / "train" / "train_source3.tsv"

def _norm_stats(df, label, n=500_000, seed=42):
    s = df.sample(n=min(n, len(df)), random_state=seed) if len(df) > n else df
    print(f"--- {label} (n={len(s):,} sampled) ---")
    for c_raw, c_norm in [("business_name", "name_norm"), ("business_address", "addr_norm")]:
        ur, un = s[c_raw].nunique(), s[c_norm].nunique()
        dr = float(s[c_raw].duplicated(keep=False).mean())
        dn = float(s[c_norm].duplicated(keep=False).mean())
        print(f"{c_raw}: nunique raw={ur:,} norm={un:,} | dup_rate raw={dr:.4f} norm={dn:.4f}")

s2 = add_norm_columns(pd.read_csv(TRAIN_S2, sep="\t", dtype="string", keep_default_na=True))
print("s2:", s2.shape, list(s2.columns))
_norm_stats(s2, "train_source2")
for _, r in s2[["business_name", "name_norm", "business_address", "addr_norm"]].sample(4, random_state=7).iterrows():
    print("-" * 100)
    print("NAME:", r["business_name"], "\n  -> norm:", r["name_norm"])
    print("ADDR:", r["business_address"], "\n  -> norm:", r["addr_norm"])

s3 = add_norm_columns(pd.read_csv(TRAIN_S3, sep="\t", dtype="string", keep_default_na=True))
print("s3:", s3.shape, list(s3.columns))
_norm_stats(s3, "train_source3")
for _, r in s3[["business_name", "name_norm", "business_address", "addr_norm"]].sample(4, random_state=9).iterrows():
    print("-" * 100)
    print("NAME:", r["business_name"], "\n  -> norm:", r["name_norm"])
    print("ADDR:", r["business_address"], "\n  -> norm:", r["addr_norm"])


s2: (5034616, 8) ['entity_id', 'business_name', 'business_address', 'country', 'name_norm', 'addr_norm', 'name_compact', 'addr_compact']
--- train_source2 (n=500,000 sampled) ---
business_name: nunique raw=479,424 norm=428,024 | dup_rate raw=0.0668 norm=0.1731
business_address: nunique raw=476,449 norm=472,361 | dup_rate raw=0.0600 norm=0.0757
----------------------------------------------------------------------------------------------------
NAME: WEBSTER APEX OMEGA, CORPORATION 
  -> norm: webster apex omega corp
ADDR: 65 PENNSYLVANIA AVENUE, LONG BEACH, NY 
  -> norm: 65 pennsylvania ave long beach ny
----------------------------------------------------------------------------------------------------
NAME: Inc Thomas and Dgvei 
  -> norm: inc thomas and dgvei
ADDR: 5619 REGENCY PARK CT, SUITLAND, MD 
  -> norm: 5619 regency park ct suitland md
----------------------------------------------------------------------------------------------------
NAME: Holdings Atlantic Project 
  -> no